# Gaia Intelligent Query Pipeline

All logic lives in `pipeline/`. This notebook is the run interface only.

In [1]:
import sys
sys.path.insert(0, './pipeline')

# Load model ONCE per session (cached — safe to re-run)
import config   # SSL fix, env vars, standard imports
import model    # loads vLLM model with cache guard

config.py loaded. HF cache: /d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache


/usr/local/lib/python3.11/dist-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading Qwen/Qwen2.5-7B-Instruct from /d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache …


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

WARNING 05-19 18:37:10 config.py:2276] Casting torch.bfloat16 to torch.float16.
INFO 05-19 18:37:22 config.py:510] This model supports multiple tasks: {'embed', 'generate', 'reward', 'classify', 'score'}. Defaulting to 'generate'.
INFO 05-19 18:37:22 llm_engine.py:234] Initializing an LLM engine (v0.6.6.post1) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forwa

...

INFO 05-19 18:37:25 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 05-19 18:37:25 selector.py:129] Using XFormers backend.
INFO 05-19 18:37:27 model_runner.py:1094] Starting to load model Qwen/Qwen2.5-7B-Instruct...
INFO 05-19 18:37:29 weight_utils.py:251] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 05-19 18:38:12 model_runner.py:1099] Loading model weights took 14.2487 GB
INFO 05-19 18:38:20 worker.py:241] Memory profiling takes 7.91 seconds
INFO 05-19 18:38:20 worker.py:241] the current vLLM instance can use total_gpu_memory (31.73GiB) x gpu_memory_utilization (0.90) = 28.56GiB
INFO 05-19 18:38:20 worker.py:241] model weights take 14.25GiB; non_torch_memory takes 0.13GiB; PyTorch activation peak memory takes 4.35GiB; the rest of the memory reserved for KV Cache is 9.82GiB.
INFO 05-19 18:38:20 gpu_executor.py:76] # GPU blocks: 11497, # CPU blocks: 4681
INFO 05-19 18:38:20 gpu_executor.py:80] Maximum concurrency for 32768 tokens per request: 5.61x
INFO 05-19 18:38:26 model_runner.py:1415] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utiliz

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:31<00:00,  1.13it/s]

INFO 05-19 18:38:57 model_runner.py:1535] Graph capturing finished in 31 secs, took 0.20 GiB
INFO 05-19 18:38:57 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 44.85 seconds
Model loaded.


In [2]:
from pipeline import routed_pipeline
from IPython.display import display

router.py loaded.
parser.py loaded.
validator.py loaded.
cost_judge.py loaded.
tap.py loaded.
simple_pipeline.py loaded.
complex_pipeline.py loaded.
pipeline.py loaded. Use routed_pipeline(query) to run.


In [4]:
# ── Choose your query ─────────────────────────────────────────────────────

# Simple queries
# USER_QUERY = 'Show me stars around the Pleiades'
# USER_QUERY = 'HR diagram of stars near Omega Centauri'
# USER_QUERY = 'Show me the red dwarfs'
# USER_QUERY = 'Variable stars near the LMC'
# USER_QUERY = 'Show me the 50 nearest stars to the Sun'
# USER_QUERY = 'Cross-match stars near the galactic centre with astrophysical parameters'

# Complex queries
# USER_QUERY = 'Show me the HR diagram and the colour histogram of the Pleiades'
# USER_QUERY = 'Compare colour distributions near Andromeda and the LMC'
# USER_QUERY = 'Find variable stars near Orion, cross-match with the Cepheid table, and plot their HR diagram'
# USER_QUERY = 'Show me red dwarfs and also white dwarfs'

USER_QUERY = 'Show me stars around the Pleiades'

# ── Run ───────────────────────────────────────────────────────────────────
try:
    result  = routed_pipeline(USER_QUERY)
    out     = result['output_json']
    results = result['results']

    s = out['summary']
    print(f"\n{'═'*60}")
    print(f"  RESULT  [{out['routed_as'].upper()} / {out['composition'].upper()}]")
    print(f"  Steps : {s['total_steps']}  ✓ {s['successful']}  ✗ {s['failed']}")
    print(f"  Time  : {s['total_duration_seconds']}s")
    print(f"{'═'*60}\n")

    for key, df in results.items():
        if df is not None:
            print(f'--- Result {key} ({len(df)} rows) ---')
            display(df.head(10))

except RuntimeError as e:
    print(f'\n🔴  {e}')


════════════════════════════════════════════════════════════
  USER QUERY: Show me stars around the Pleiades
════════════════════════════════════════════════════════════

[Router] Classifying query …


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.24it/s, est. speed input: 843.62 toks/s, output: 35.90 toks/s]


  🟢  Verdict: SIMPLE
      Reason: single cone search


  USER QUERY: Show me stars around the Pleiades

[Attempt 1/3] Parsing …


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.60s/it, est. speed input: 877.45 toks/s, output: 36.92 toks/s]


  Parsed JSON:
{
  "intent": "cone_search",
  "ra": 56.75,
  "dec": 24.12,
  "radius": 1.0,
  "columns": [
    "source_id",
    "ra",
    "dec",
    "phot_g_mean_mag"
  ],
  "filters": {},
  "join_table": null,
  "limit": 1000
}
  JSON valid.
  ADQL: SELECT TOP 1000 source_id, ra, dec, phot_g_mean_mag, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 56.75, 24.12, 1.0)) ORDER BY random_index
  ADQL syntax valid.
  Evaluating cost …


Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.47s/it, est. speed input: 229.85 toks/s, output: 43.30 toks/s]


  ────────────────────────────────────────────────────
  🟢  COST: CHEAP  (score 20/100)
  ────────────────────────────────────────────────────
    ⚠️  TOP ≤ 1000
    ⚠️  Tight cone ≤ 1.0°
    ⚠️  Indexed filter (spatial)
    ⚠️  No JOIN
    ⚠️  ORDER BY on random_index (not indexed, but not a cost driver)
    💡 No further optimisations needed
  ────────────────────────────────────────────────────

Query sent:
SELECT TOP 1000 source_id, ra, dec, phot_g_mean_mag, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 56.75, 24.12, 1.0)) ORDER BY random_index

INFO: Query finished. [astroquery.utils.tap.core]
Got 1,000 rows

  ✓  1000 rows in 7.23s
  Output JSON → ./pipeline_outputs/show_me_stars_around_the_pleiades.json

════════════════════════════════════════════════════════════
  RESULT  [SIMPLE / SIMPLE]
  Steps : 1  ✓ 1  ✗ 0
  Time  : 7.23s
════════════════════════════════════════════════════════════

--- Result 1 (1000 rows) ---


,source_id,ra,dec,phot_g_mean_mag,bp_rp,pmra,pmdec
0,65170528079945984,56.401759,23.518676,19.054289,2.064264,3.077967,-2.793100
1,69817201658236288,56.514012,24.927902,18.234938,1.648279,1.368850,-1.605669
2,66506331628024832,57.300875,23.886588,8.101668,0.296336,19.620729,-45.173239
3,68269471539578240,55.879030,24.302657,15.825923,1.156813,5.913426,-1.465584
4,65266700987253504,55.927350,24.262811,19.298132,2.836407,17.480791,-12.895002
5,66835192980249600,56.846262,24.897098,20.599745,0.625975,1.194813,-1.848215
6,66529425671692160,57.251529,24.093894,20.367111,1.491135,9.185393,-21.455008
7,66732727945184384,56.635586,24.278700,18.735273,1.180223,0.318564,-0.439097
8,66831099877023232,57.177749,25.013688,18.444952,0.913834,1.286902,-1.040130
9,66794476691711616,56.663395,24.632537,18.547205,0.968925,-0.171548,-1.143023
